#Lab — Build a News Digest Tool

Step 1 — Get sample articles


In [1]:
articles = [
"""Apple unveiled its newest iPhone at an event in Cupertino on
Tuesday.
CEO Tim Cook said the device features a faster chip and improved
camera.
The phone will be available in stores starting next month, the company
said, with prices starting at $999.""",
"""The World Health Organization announced new guidelines on Monday
aimed at reducing sugar consumption worldwide. Officials in Geneva
said
the changes could prevent millions of cases of diabetes over the next
decade if adopted by national governments.""",
"""Tesla reported record deliveries for the third quarter, beating
analyst expectations. The Texas-based company said demand remained
strong in China and Europe despite increased competition from
traditional automakers entering the EV market.""",
"""NASA's Perseverance rover discovered new evidence of ancient
water activity on Mars, according to a paper published this week.
Scientists at the Jet Propulsion Laboratory said the findings
strengthen the case that the planet once had a wetter climate.""",
"""Manchester United announced the signing of a new midfielder
from Barcelona for a reported fee of 80 million euros. The 24-year-old
player will join the club ahead of the new season starting in
August.""",
]

Step 2 — Load models once

In [2]:
import spacy
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load each model ONCE — reused for every article
nlp = spacy.load('en_core_web_sm')

# Direct loading for summarization model
summarizer_tokenizer = AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6")
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Step 3 — Process each article

In [3]:
def digest_article(text):
    # Summarise (abstractive, short)
    inputs = summarizer_tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    summary_ids = summarizer_model.generate(inputs["input_ids"], num_beams=4, max_length=40, min_length=10, do_sample=False, early_stopping=True)
    summary_text = summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # Extract entities
    doc = nlp(text)
    entities = {}

    for ent in doc.ents:
        entities.setdefault(ent.label_, set()).add(ent.text)

    return summary_text, entities


for i, article in enumerate(articles, 1):
    summary, entities = digest_article(article)

    print(f"--- Article {i} ---")
    print("Summary :", summary)
    print("Entities:")

    for label, names in entities.items():
        print(f"  {label}: {', '.join(names)}")

    print()

--- Article 1 ---
Summary :  CEO Tim Cook said the device features a faster chip and improved camera . The phone will be available in stores starting next month, with prices starting at $999 .
Entities:
  ORG: Apple
  GPE: Cupertino
  DATE: Tuesday, next month
  PERSON: Tim Cook
  MONEY: 999

--- Article 2 ---
Summary :  The World Health Organization announced new guidelines aimed at reducing sugar consumption worldwide . Officials in Geneva said the changes could prevent millions of cases of diabetes over the next decade .
Entities:
  ORG: The World Health Organization
  DATE: Monday, the next
decade
  GPE: Geneva
  CARDINAL: millions

--- Article 3 ---
Summary :  Tesla reported record deliveries for the third quarter, beating analyst expectations . The Texas-based company said demand remained strong in China and Europe .
Entities:
  DATE: the third quarter
  GPE: China, Texas
  LOC: Europe
  ORG: EV

--- Article 4 ---
Summary :  NASA's Perseverance rover discovered new evidence of an